In [2]:
# verify imported data matches numbers
import pandas as pd

df = pd.read_pickle("../data/flagged.pkl") # load flagged data into a table

print(df.shape)
print(df.columns.tolist())

has_issue = df["issue_count"] >= 1
print("Numebr of rows without issues:", (len(df)-has_issue.sum()))
print("Percentage of rows with issue(s):", ((has_issue.sum()/len(df))* 100).round(2))

(1067371, 17)
['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price', 'customer_id', 'country', 'is_duplicate', 'is_cancellation', 'missing_customer', 'missing_description', 'bad_quantity', 'bad_price', 'non_product_code', 'issue_count', 'line_total']
Numebr of rows without issues: 776594
Percentage of rows with issue(s): 27.24


In [ ]:
# Order	Rule	        Destination	     Reason
# 1	Duplicate	        Reject	        A copy of a row already counted
# 2	Non-product code    Reject	        Fees and adjustments, not merchandise, even when cancelled
# 3	Cancellation	    Cancellations	Real product returns, tracked separately
# 4	Bad quantity	    Reject	        Stock write-offs, not sales
# 5	Bad price	        Reject	        No money changed hands
# —	Everything else	    Sales	        Includes missing customer and missing description

# turn these rules into column "outcome"

import numpy as np # used to checl conditions

conditions = [
    df["is_duplicate"],
    df["non_product_code"],
    df["is_cancellation"],
    df["bad_quantity"],
    df["bad_price"],
]
outcomes = [
    "rejected: duplicate",
    "rejected: non product code",
    "cancellation",
    "rejected: stock adjustement",
    "rejected: bad price",
]

df["outcome"] = np.select(conditions, outcomes, default="sales")
df["outcome"].value_counts() # shows how many rows got each outcome

print(df["outcome"].value_counts().sum() == len(df)) # check if total rows of outcome equald total rows

True
